# Notebook 01: Exploratory Data Analysis & Data Profiling

## CRISP-DM Phases Covered: Business Understanding, Data Understanding, Data Preparation

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Reference**: Petropoulos et al. (2022) *Forecasting: theory and practice*, International Journal of Forecasting, 38(3), 705–871.  
**EDA Methodology**: CRISP-DM / CRISP-ML(Q) framework as taught in Advanced EDA module.

---

### Business Understanding

OptiWMS aims to optimise warehouse slotting through demand forecasting and genetic algorithm placement. The core business KPIs are:

| KPI | Target | Metric |
|-----|--------|--------|
| Forecast accuracy | WAPE < 0.10 | Weighted Absolute Percentage Error |
| Stockout reduction | < 3% of SKU-months | Zero-demand events due to supply failure |
| Fill rate | > 95% | Orders fulfilled from available stock |
| Slotting efficiency | Minimise pick path | GA fitness score (lower = better) |

### Data Sources

1. **Finished Goods (FG)**: 103 FMCG SKUs, 36 months, synthetic but FMCG-realistic demand (2k–110k units/month)  
   - Source: `hemas_scenario_c_dataset_cleaned.csv`
2. **Raw Materials (RM)**: 288 chemical/packaging SKUs, 36 months, anchored to real planning parameters  
   - Source: `rule_based_wms_monthly.csv` (generated from `active_stock_canonical.csv`)
3. **Product Dimensions**: 391 SKUs with L/W/H, weight, storage type  
   - Source: `product_dimensions.csv`

Since this is a university project, real historical data is unavailable. Synthetic data is anchored to real planning parameters (MOQ, ROP, buffer days) from an actual FMCG manufacturer.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False
    print('missingno not installed — pip install missingno')

try:
    from statsmodels.tsa.seasonal import STL
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels not installed — pip install statsmodels')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

SEED = 42
np.random.seed(SEED)

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'Forecast model train data optiwms'
GEN_DIR  = ROOT / 'outputs' / 'generated'

print(f'Data directory: {DATA_DIR}')
print(f'Generated directory: {GEN_DIR}')

## 1. Data Loading and Initial Inspection

> *"The first step in any forecasting exercise is to understand the data at hand."*  
> — Petropoulos et al. (2022), Section 2.2

In [ ]:
# Load FG demand data (Scenario C — 103 FMCG products)
fg_path = DATA_DIR / 'hemas_scenario_c_dataset_cleaned.csv'
fg = pd.read_csv(fg_path)
fg['month'] = pd.to_datetime(fg['month'])
if 'demand_units_clean' in fg.columns:
    fg['demand_units'] = fg['demand_units_clean']

# Load RM demand data (288 raw material SKUs)
rm_path = GEN_DIR / 'rule_based_wms_monthly.csv'
rm = pd.read_csv(rm_path)
rm['month'] = pd.to_datetime(rm['month'])

# Load product dimensions
dims_path = GEN_DIR / 'product_dimensions.csv'
dims = pd.read_csv(dims_path)

print('=== Finished Goods (FG) ===')
print(f'Shape: {fg.shape}')
print(f'SKUs: {fg["fg_code"].nunique()}')
print(f'Date range: {fg["month"].min()} to {fg["month"].max()}')
print(f'Months: {fg["month"].nunique()}')
print()
print('=== Raw Materials (RM) ===')
print(f'Shape: {rm.shape}')
print(f'SKUs: {rm["fg_code"].nunique()}')
print(f'Date range: {rm["month"].min()} to {rm["month"].max()}')
print()
print('=== Product Dimensions ===')
print(f'Shape: {dims.shape}')
print(f'FG: {(dims["sku_type"]=="FG").sum()}, RM: {(dims["sku_type"]=="RM").sum()}')
print()
print('--- FG Sample ---')
fg.head(3)

In [ ]:
# Data types and basic info
print('=== FG Data Types ===')
print(fg.dtypes)
print()
print('=== FG Descriptive Statistics ===')
fg.describe()

## 2. Data Cleaning & Validation

Following the CRISP-DM data preparation phase, we systematically check for:
- Missing values
- Duplicates
- Invalid data types
- Outliers

> *"Thorough cleaning prevents the 'garbage in, garbage out' problem."* — EDA Module

In [ ]:
# 2.1 Missing Value Analysis
print('=== Missing Values (FG) ===')
fg_nulls = fg.isnull().sum()
fg_nulls_pct = (fg_nulls / len(fg) * 100).round(2)
null_report = pd.DataFrame({'null_count': fg_nulls, 'null_pct': fg_nulls_pct})
print(null_report[null_report['null_count'] > 0] if null_report['null_count'].sum() > 0 else 'No missing values in FG data.')
print()

print('=== Missing Values (RM) ===')
rm_nulls = rm.isnull().sum()
rm_nulls_pct = (rm_nulls / len(rm) * 100).round(2)
null_report_rm = pd.DataFrame({'null_count': rm_nulls, 'null_pct': rm_nulls_pct})
print(null_report_rm[null_report_rm['null_count'] > 0] if null_report_rm['null_count'].sum() > 0 else 'No missing values in RM data.')

# Missingno matrix visualization
if HAS_MISSINGNO:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    plt.sca(axes[0])
    msno.matrix(fg.sample(min(500, len(fg)), random_state=SEED), ax=axes[0], sparkline=False)
    axes[0].set_title('FG Data — Missing Value Matrix', fontsize=13)
    plt.sca(axes[1])
    msno.matrix(rm.sample(min(500, len(rm)), random_state=SEED), ax=axes[1], sparkline=False)
    axes[1].set_title('RM Data — Missing Value Matrix', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Install missingno for visualization: pip install missingno')

In [ ]:
# 2.2 Duplicate Check
fg_dupes = fg.duplicated(subset=['month', 'fg_code']).sum()
rm_dupes = rm.duplicated(subset=['month', 'fg_code']).sum()
print(f'FG duplicate (month, fg_code) pairs: {fg_dupes}')
print(f'RM duplicate (month, fg_code) pairs: {rm_dupes}')

# 2.3 Data Type Validation
print('\n=== Type Validation ===')
expected_numeric = ['demand_units', 'on_hand_inventory', 'lead_time_days', 'supplier_otif']
for col in expected_numeric:
    if col in fg.columns:
        non_numeric = pd.to_numeric(fg[col], errors='coerce').isna().sum() - fg[col].isna().sum()
        print(f'  FG.{col}: {"OK" if non_numeric == 0 else f"{non_numeric} non-numeric values"}')

# 2.4 Negative value check
print('\n=== Negative Value Check (FG) ===')
for col in ['demand_units', 'on_hand_inventory', 'lead_time_days']:
    if col in fg.columns:
        neg_count = (fg[col] < 0).sum()
        print(f'  {col}: {neg_count} negative values ({"OK" if neg_count == 0 else "NEEDS ATTENTION"})')

In [ ]:
# 2.5 Outlier Detection using IQR method (EDA Zuu slide 19)
def detect_outliers_iqr(series, multiplier=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    outliers = ((series < lower) | (series > upper)).sum()
    return outliers, lower, upper

print('=== Outlier Detection (IQR Method, 1.5x) ===')
print(f'{"Column":<25} {"Outliers":>10} {"Lower":>12} {"Upper":>12} {"% Outlier":>10}')
print('-' * 70)
for col in ['demand_units', 'on_hand_inventory', 'lead_time_days', 'stockout_days']:
    if col in fg.columns:
        n_out, lo, hi = detect_outliers_iqr(fg[col].dropna())
        pct = n_out / len(fg) * 100
        print(f'{col:<25} {n_out:>10} {lo:>12.1f} {hi:>12.1f} {pct:>9.2f}%')

# Box plot visualization of outliers
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(['demand_units', 'on_hand_inventory', 'lead_time_days', 'stockout_days']):
    if col in fg.columns:
        sns.boxplot(y=fg[col], ax=axes[i])
        axes[i].set_title(f'{col}', fontsize=11)
        axes[i].set_ylabel('')
plt.suptitle('FG Data — Outlier Detection (Box Plots)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Univariate Analysis

> *"Examine distribution of individual variables: central tendency, spread, and shape."* — EDA Module, Slide 22

We analyse each key variable independently using histograms, density plots, and summary statistics including **skewness** and **kurtosis**.

In [ ]:
# 3.1 Demand Distribution — Histogram + KDE
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# FG demand distribution
ax = axes[0, 0]
sns.histplot(fg['demand_units'], bins=60, kde=True, ax=ax)
ax.axvline(fg['demand_units'].mean(), color='red', linestyle='--', label=f'Mean: {fg["demand_units"].mean():,.0f}')
ax.axvline(fg['demand_units'].median(), color='green', linestyle='--', label=f'Median: {fg["demand_units"].median():,.0f}')
ax.set_title('FG Demand Distribution')
ax.legend()

# RM demand distribution
ax = axes[0, 1]
sns.histplot(rm['demand_units'], bins=60, kde=True, ax=ax)
ax.axvline(rm['demand_units'].mean(), color='red', linestyle='--', label=f'Mean: {rm["demand_units"].mean():,.0f}')
ax.axvline(rm['demand_units'].median(), color='green', linestyle='--', label=f'Median: {rm["demand_units"].median():,.0f}')
ax.set_title('RM Demand Distribution')
ax.legend()

# FG demand by category (violin)
ax = axes[1, 0]
top_cats = fg['fg_category'].value_counts().head(6).index
sns.boxplot(data=fg[fg['fg_category'].isin(top_cats)], x='fg_category', y='demand_units', ax=ax)
ax.set_title('FG Demand by Category')
ax.tick_params(axis='x', rotation=30)

# Log-transformed demand
ax = axes[1, 1]
fg['demand_log'] = np.log1p(fg['demand_units'])
sns.histplot(fg['demand_log'], bins=50, kde=True, ax=ax)
ax.set_title('FG Demand (log1p transformed)')
ax.set_xlabel('log1p(demand_units)')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Summary Statistics Table with Skewness and Kurtosis
numeric_cols = ['demand_units', 'on_hand_inventory', 'stockout_days',
                'lead_time_days', 'supplier_otif', 'inbound_po_qty',
                'open_sales_orders', 'returns_qty']
numeric_cols = [c for c in numeric_cols if c in fg.columns]

summary = fg[numeric_cols].agg(['count', 'mean', 'std', 'min', 'median', 'max',
                                 lambda x: x.skew(), lambda x: x.kurtosis()]).T
summary.columns = ['Count', 'Mean', 'Std', 'Min', 'Median', 'Max', 'Skewness', 'Kurtosis']
summary = summary.round(2)

print('=== FG Descriptive Statistics with Shape Metrics ===')
print('(Skewness > 1 = highly right-skewed; Kurtosis > 3 = heavy-tailed)')
summary

## 4. Bivariate Analysis

> *"Bivariate analysis reveals relationships that aren't visible when examining variables in isolation."* — EDA Module, Slide 23

In [ ]:
# 4.1 Correlation Heatmap (all numeric features)
corr_cols = [c for c in numeric_cols if c in fg.columns]
corr_matrix = fg[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            linewidths=0.5)
ax.set_title('FG Data — Pearson Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Highlight strong correlations
print('\n=== Strong Correlations (|r| > 0.3) ===')
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            print(f'  {corr_matrix.index[i]} <-> {corr_matrix.columns[j]}: r = {r:.3f}')

In [ ]:
# 4.2 Demand vs Promotion and Holiday (Bivariate categorical-numerical)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Demand by promotion flag
if 'promotion_flag' in fg.columns:
    sns.boxplot(data=fg, x='promotion_flag', y='demand_units', ax=axes[0])
    axes[0].set_title('Demand by Promotion Flag')
    axes[0].set_xticklabels(['No Promotion', 'Promotion'])

# Demand by holiday flag
if 'holiday_flag' in fg.columns:
    sns.violinplot(data=fg, x='holiday_flag', y='demand_units', ax=axes[1])
    axes[1].set_title('Demand by Holiday Flag')
    axes[1].set_xticklabels(['Non-Holiday', 'Holiday'])

# Demand vs lead time scatter
if 'lead_time_days' in fg.columns:
    sns.scatterplot(data=fg.sample(min(2000, len(fg)), random_state=SEED),
                    x='lead_time_days', y='demand_units', alpha=0.3, ax=axes[2])
    axes[2].set_title('Demand vs Lead Time')

plt.suptitle('Bivariate Analysis — Demand Relationships', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Multivariate Analysis

> *"How do variables interact together? Are there complex relationships that simple correlations miss?"* — EDA Module, Slide 24

In [ ]:
# 5.1 Pair Plot (sampled for performance)
pair_cols = ['demand_units', 'on_hand_inventory', 'lead_time_days', 'stockout_days']
pair_cols = [c for c in pair_cols if c in fg.columns]

sample = fg[pair_cols + ['fg_category']].sample(min(1500, len(fg)), random_state=SEED)
top3 = fg['fg_category'].value_counts().head(3).index.tolist()
sample_filtered = sample[sample['fg_category'].isin(top3)]

g = sns.pairplot(sample_filtered, hue='fg_category', diag_kind='kde',
                 plot_kws={'alpha': 0.4, 's': 20}, height=2.5)
g.figure.suptitle('Pair Plot — Top 3 FG Categories', y=1.02, fontsize=14)
plt.show()

In [ ]:
# 5.2 PCA — Dimensionality Reduction Visualization
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca_cols = [c for c in numeric_cols if c in fg.columns]
X_pca = fg[pca_cols].dropna()
X_scaled = StandardScaler().fit_transform(X_pca)

pca = PCA(n_components=min(5, len(pca_cols)))
components = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Explained variance
cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_,
            alpha=0.7, label='Individual')
axes[0].plot(range(1, len(cum_var)+1), cum_var, 'ro-', label='Cumulative')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA — Explained Variance')
axes[0].legend()
axes[0].axhline(y=0.90, color='gray', linestyle='--', alpha=0.5)

# PC1 vs PC2 scatter
cat_map = fg.loc[X_pca.index, 'fg_category']
for cat in top3:
    mask = cat_map == cat
    axes[1].scatter(components[mask, 0], components[mask, 1], alpha=0.3, s=15, label=cat)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('PCA — PC1 vs PC2 by Category')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Components needed for 90% variance: {np.argmax(cum_var >= 0.90) + 1}')

## 6. Time Series Analysis

> *"Pre-processing data: time series decomposition separates trend, seasonality, and residual components."*  
> — Petropoulos et al. (2022), Section 2.2.2

In [ ]:
# 6.1 Aggregate demand over time
fg_monthly = fg.groupby('month')['demand_units'].agg(['sum', 'mean', 'std']).reset_index()
fg_monthly.columns = ['month', 'total_demand', 'avg_demand', 'std_demand']

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(fg_monthly['month'], fg_monthly['total_demand'], 'b-o', markersize=4)
axes[0].fill_between(fg_monthly['month'],
                      fg_monthly['total_demand'] - fg_monthly['std_demand'] * fg['fg_code'].nunique()**0.5,
                      fg_monthly['total_demand'] + fg_monthly['std_demand'] * fg['fg_code'].nunique()**0.5,
                      alpha=0.2)
axes[0].set_title('FG Total Demand Over Time (with 1-sigma band)')
axes[0].set_ylabel('Total Demand (units)')

# Individual SKU time series (sample 8 SKUs)
sample_skus = fg['fg_code'].unique()[:8]
for sku in sample_skus:
    sku_data = fg[fg['fg_code'] == sku].sort_values('month')
    axes[1].plot(sku_data['month'], sku_data['demand_units'], alpha=0.6, linewidth=1, label=sku)
axes[1].set_title('Individual SKU Demand Trajectories (8 samples)')
axes[1].set_ylabel('Demand (units)')
axes[1].legend(ncol=4, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 6.2 STL Decomposition (Petropoulos Section 2.2.2)
if HAS_STATSMODELS:
    # Pick a high-volume SKU for clear decomposition
    top_sku = fg.groupby('fg_code')['demand_units'].mean().idxmax()
    sku_ts = fg.loc[fg['fg_code'] == top_sku, ['month', 'demand_units']].copy()
    # Normalize to month-start and collapse duplicate month labels
    sku_ts['month'] = sku_ts['month'].dt.to_period('M').dt.to_timestamp()
    ts = sku_ts.groupby('month')['demand_units'].sum().sort_index()
    ts = ts.asfreq('MS')  # monthly start frequency

    stl = STL(ts, period=12, robust=True)
    result = stl.fit()

    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    axes[0].plot(ts.index, ts.values)
    axes[0].set_title(f'STL Decomposition — {top_sku} ({fg[fg["fg_code"]==top_sku]["fg_name"].iloc[0]})')
    axes[0].set_ylabel('Observed')

    axes[1].plot(ts.index, result.trend)
    axes[1].set_ylabel('Trend')

    axes[2].plot(ts.index, result.seasonal)
    axes[2].set_ylabel('Seasonal')

    axes[3].plot(ts.index, result.resid)
    axes[3].axhline(0, color='gray', linestyle='--')
    axes[3].set_ylabel('Residual')

    plt.tight_layout()
    plt.show()

    strength_seasonal = 1 - result.resid.var() / (result.seasonal + result.resid).var()
    strength_trend = 1 - result.resid.var() / (result.trend + result.resid).var()
    print(f'Seasonal strength: {strength_seasonal:.3f} (> 0.6 = significant seasonality)')
    print(f'Trend strength: {strength_trend:.3f}')
else:
    print('Install statsmodels for STL decomposition')

In [ ]:
# 6.3 ACF and PACF plots (guides ARIMA order selection)
if HAS_STATSMODELS:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    plot_acf(ts, lags=24, ax=axes[0], title=f'ACF — {top_sku}')
    plot_pacf(ts, lags=12, ax=axes[1], title=f'PACF — {top_sku}', method='ywm')
    plt.tight_layout()
    plt.show()
    
    print('ACF interpretation:')
    print('  - Slow decay in ACF suggests non-stationarity or trend')
    print('  - Significant spike at lag 12 in ACF suggests yearly seasonality')
    print('  - PACF cuts off after lag p -> suggests AR(p) component')

## 7. Intermittent Demand Analysis

> *"Intermittent demand is characterised by infrequent demand arrivals [...] Croston's method and its variants remain the standard approach."*  
> — Petropoulos et al. (2022), Section 2.8

For warehouse RM SKUs, many items have sporadic demand. We classify SKUs by their **zero-demand rate** and **average inter-demand interval (ADI)** to determine appropriate forecasting methods.

In [ ]:
# 7.1 Zero-demand rate per SKU
rm_zero = rm.groupby('fg_code').agg(
    zero_rate=('demand_units', lambda x: (x == 0).mean()),
    mean_demand=('demand_units', 'mean'),
    cv=('demand_units', lambda x: x.std() / max(x.mean(), 1)),
).reset_index()

# Demand classification (Syntetos-Boylan framework, referenced in Petropoulos Section 2.8.3)
# ADI > 1.32 and CV^2 > 0.49 -> Lumpy
# ADI > 1.32 and CV^2 <= 0.49 -> Intermittent
# ADI <= 1.32 and CV^2 > 0.49 -> Erratic
# ADI <= 1.32 and CV^2 <= 0.49 -> Smooth
rm_zero['cv2'] = rm_zero['cv'] ** 2
rm_zero['adi'] = 1 / (1 - rm_zero['zero_rate'] + 1e-6)

def classify_demand(row):
    if row['adi'] > 1.32 and row['cv2'] > 0.49:
        return 'Lumpy'
    elif row['adi'] > 1.32:
        return 'Intermittent'
    elif row['cv2'] > 0.49:
        return 'Erratic'
    return 'Smooth'

rm_zero['demand_class'] = rm_zero.apply(classify_demand, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Syntetos-Boylan classification scatter
for cls in ['Smooth', 'Erratic', 'Intermittent', 'Lumpy']:
    mask = rm_zero['demand_class'] == cls
    axes[0].scatter(rm_zero.loc[mask, 'cv2'], rm_zero.loc[mask, 'adi'],
                    alpha=0.5, s=30, label=f'{cls} ({mask.sum()})')
axes[0].axhline(1.32, color='red', linestyle='--', alpha=0.5)
axes[0].axvline(0.49, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('CV² (demand variability)')
axes[0].set_ylabel('ADI (average demand interval)')
axes[0].set_title('Syntetos-Boylan Demand Classification (RM SKUs)')
axes[0].legend()

# Distribution of zero-demand rates
axes[1].hist(rm_zero['zero_rate'], bins=30, edgecolor='black')
axes[1].axvline(0.3, color='red', linestyle='--', label='Intermittent threshold (30%)')
axes[1].set_xlabel('Zero-Demand Rate')
axes[1].set_ylabel('Number of SKUs')
axes[1].set_title('RM SKU Zero-Demand Rate Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n=== Demand Classification Summary (Syntetos-Boylan) ===')
print(rm_zero['demand_class'].value_counts())
print(f'\nSKUs needing Croston/SBA (Intermittent + Lumpy): {(rm_zero["demand_class"].isin(["Intermittent", "Lumpy"])).sum()}')
print(f'SKUs suitable for ETS/ML (Smooth + Erratic): {(rm_zero["demand_class"].isin(["Smooth", "Erratic"])).sum()}')

## 8. ABC / FMS Classification Analysis

ABC analysis classifies products by **value contribution** (Pareto principle: 80/15/5).  
FMS classifies by **movement frequency** (Fast/Medium/Slow).  
The **amalgamated ABC-FMS matrix** determines optimal warehouse zone placement.

In [ ]:
# 8.1 ABC Classification — Pareto Analysis
fg_annual = fg.groupby('fg_code')['demand_units'].sum().sort_values(ascending=False).reset_index()
fg_annual.columns = ['fg_code', 'annual_demand']
fg_annual['cumulative_pct'] = fg_annual['annual_demand'].cumsum() / fg_annual['annual_demand'].sum() * 100

def assign_abc(cum_pct):
    if cum_pct <= 80:
        return 'A'
    elif cum_pct <= 95:
        return 'B'
    return 'C'

fg_annual['abc_class'] = fg_annual['cumulative_pct'].apply(assign_abc)

# FMS Classification
fg_velocity = fg.groupby('fg_code')['demand_units'].mean().reset_index()
fg_velocity.columns = ['fg_code', 'avg_monthly']
q_fast = fg_velocity['avg_monthly'].quantile(0.7)
q_slow = fg_velocity['avg_monthly'].quantile(0.3)

def assign_fms(avg):
    if avg >= q_fast:
        return 'Fast'
    elif avg >= q_slow:
        return 'Medium'
    return 'Slow'

fg_velocity['fms_class'] = fg_velocity['avg_monthly'].apply(assign_fms)

# Merge
classification = fg_annual.merge(fg_velocity, on='fg_code')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Pareto chart
ax1 = axes[0]
colors = {'A': '#e74c3c', 'B': '#f39c12', 'C': '#27ae60'}
bars = ax1.bar(range(len(fg_annual)), fg_annual['annual_demand'],
               color=[colors[c] for c in fg_annual['abc_class']], alpha=0.7)
ax2 = ax1.twinx()
ax2.plot(range(len(fg_annual)), fg_annual['cumulative_pct'], 'k-o', markersize=2)
ax2.axhline(80, color='red', linestyle='--', alpha=0.5)
ax2.axhline(95, color='orange', linestyle='--', alpha=0.5)
ax1.set_xlabel('SKU Rank')
ax1.set_ylabel('Annual Demand')
ax2.set_ylabel('Cumulative %')
ax1.set_title('ABC Pareto Analysis')

# ABC distribution
abc_counts = classification['abc_class'].value_counts().reindex(['A', 'B', 'C'])
axes[1].bar(abc_counts.index, abc_counts.values, color=['#e74c3c', '#f39c12', '#27ae60'])
for i, (idx, val) in enumerate(abc_counts.items()):
    axes[1].text(i, val + 1, str(val), ha='center', fontweight='bold')
axes[1].set_title('ABC Distribution')
axes[1].set_ylabel('Number of SKUs')

# Amalgamated ABC-FMS heatmap
cross = pd.crosstab(classification['abc_class'], classification['fms_class'])
cross = cross.reindex(index=['A', 'B', 'C'], columns=['Fast', 'Medium', 'Slow'], fill_value=0)
sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd', ax=axes[2])
axes[2].set_title('ABC-FMS Amalgamated Matrix')
axes[2].set_ylabel('ABC Class')
axes[2].set_xlabel('FMS Class')

plt.tight_layout()
plt.show()

print('\n=== Classification Summary ===')
print(f'A-class SKUs: {abc_counts.get("A", 0)} ({abc_counts.get("A", 0)/len(classification)*100:.0f}%)')
print(f'B-class SKUs: {abc_counts.get("B", 0)} ({abc_counts.get("B", 0)/len(classification)*100:.0f}%)')
print(f'C-class SKUs: {abc_counts.get("C", 0)} ({abc_counts.get("C", 0)/len(classification)*100:.0f}%)')
print(f'\nFast movers: {(classification["fms_class"]=="Fast").sum()}')
print(f'Slow movers: {(classification["fms_class"]=="Slow").sum()}')

## 9. Product Dimensions & Storage Type Profile

In [ ]:
# 9.1 Storage type distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Storage type counts
st_counts = dims['storage_type'].value_counts()
axes[0].barh(st_counts.index, st_counts.values)
axes[0].set_title('Storage Type Distribution (391 SKUs)')
axes[0].set_xlabel('Count')

# Weight distribution by storage type
sns.boxplot(data=dims, x='storage_type', y='weight_kg', ax=axes[1],
            order=['IBC', 'DRUM', 'BAG', 'ROLL', 'CARTON'])
axes[1].set_title('Weight by Storage Type')
axes[1].tick_params(axis='x', rotation=30)

# Volume distribution by type
sns.boxplot(data=dims, x='sku_type', y='volume_cm3', ax=axes[2])
axes[2].set_title('Volume: FG vs RM')

plt.tight_layout()
plt.show()

# Hazard class summary
print('\n=== Hazard Class Distribution ===')
print(dims['hazard_class'].value_counts())

## 10. Feature Summary & Transformation Journey

> *"Document: feature dictionary with descriptions, transformation steps and rationale, statistical summaries before/after, key insights, known limitations."* — EDA Module, Slide 34

### Key EDA Findings

| Finding | Implication | Action |
|---------|-------------|--------|
| FG demand is right-skewed | Log transformation may help ML models | Apply log1p in feature engineering |
| Strong seasonality detected via STL | Calendar features and seasonal models needed | Include month, quarter features |
| Promotion increases demand ~15-25% | Promotion flag is a strong predictor | Include as feature |
| RM SKUs have intermittent demand | Croston/SBA needed for sparse SKUs | Use Syntetos-Boylan to route models |
| ABC: ~20% SKUs = 80% volume | Focus ML effort on A-class items | Separate model strategy by class |
| Storage types vary widely | GA must respect storage-type constraints | Pass to fitness function |

In [ ]:
# 10.1 Feature dictionary
feature_dict = []
for col in fg.columns:
    dtype = str(fg[col].dtype)
    nulls = fg[col].isnull().sum()
    unique = fg[col].nunique()
    if np.issubdtype(fg[col].dtype, np.number):
        feature_dict.append({
            'Feature': col, 'Type': dtype, 'Nulls': nulls,
            'Unique': unique, 'Min': fg[col].min(), 'Max': fg[col].max(),
            'Mean': round(fg[col].mean(), 2)
        })
    else:
        feature_dict.append({
            'Feature': col, 'Type': dtype, 'Nulls': nulls,
            'Unique': unique, 'Min': '-', 'Max': '-', 'Mean': '-'
        })

feature_df = pd.DataFrame(feature_dict)
print('=== FG Dataset Feature Dictionary ===')
feature_df

In [ ]:
print('='*60)
print('  EDA COMPLETE — Notebook 01')
print('='*60)
print(f'\nDatasets profiled:')
print(f'  FG: {fg.shape[0]:,} rows x {fg.shape[1]} cols ({fg["fg_code"].nunique()} SKUs)')
print(f'  RM: {rm.shape[0]:,} rows x {rm.shape[1]} cols ({rm["fg_code"].nunique()} SKUs)')
print(f'  Dimensions: {dims.shape[0]} products')
print(f'\nNext: Notebook 02 — Feature Engineering')